# 20. AI 서버: 백엔드가 부를 수 있는 API로 만든 19번 (FastAPI)
19번의 기능(영역 분할 → 벽 색 + 바닥재 합성 → OpenAI 키워드 분석 → CLIP 바닥재 선택)을 **HTTP API 서버**로 감싼 노트북. 백엔드 서버가 이 서버를 불러서 쓰는 구조 (앱 → 백엔드 → AI 서버)

```
사진 업로드 → [POST /rooms] 영역 분할 1번 → room_id 기억
(선택) 붓 수정 → [POST /rooms/{room_id}/edits]
(선택) 레퍼런스 분석 → [POST /recommend] → 키워드 + 시안(벽 hex + 바닥재 id)
시안 생성 → [POST /rooms/{room_id}/generate] → 이미지 (슬라이더 값 options로 몇 번이든 다시 호출)
```
- 서버를 켜면 **`주소/docs`에서 API 명세를 바로 보고 실행**해 볼 수 있음 (백엔드·프론트 공유용)
- 인증: 요청 헤더 `X-AI-Key: <비밀 값>` (Colab 🔑에 `AI_SERVER_KEY`로 저장, 백엔드와 같은 값을 공유). `/health`만 인증 없이 열림. 🔑에 없으면 인증 없이 열리니 개발 중에만 사용
- 에러 형식은 백엔드와 같음: `{"error": {"code": "...", "message": "...", "details": null}}`
- 방(room)은 서버 메모리에 최근 30개만 기억함. 서버를 다시 켜면 사라지므로 `room_not_found`(404)가 오면 `POST /rooms`로 다시 올리면 됨
- 사진은 파일 업로드(`image`)나 주소(`image_url`) 둘 다 가능. 휴대폰 사진의 회전 정보(EXIF)를 반영함

| 요청 | 하는 일 |
|---|---|
| `GET /health` | 연결 확인 (`{"status":"ok", ...}`) |
| `POST /floors/sync` | 백엔드 바닥재 목록(`id`, `url`, 선택 `vec`)을 AI 서버에 등록 (바닥재가 바뀔 때 호출) |
| `GET /floors` | 등록된 바닥재 확인 |
| `POST /rooms` | 방 사진 올리기 → 영역 분할 → `room_id`, 영역 목록 |
| `GET /rooms/{room_id}` | 방 정보 |
| `GET /rooms/{room_id}/regions?kind=overlay\|ids` | 영역 그림 (overlay: 색칠, ids: 영역 번호가 값인 회색 그림) |
| `POST /rooms/{room_id}/edits` | 붓 수정 `{"strokes":[{"region_id":1,"width":40,"points":[[x,y],...]}]}` (좌표는 원본 사진 픽셀, region_id 0은 지우개) |
| `POST /rooms/{room_id}/reset` | 붓 수정을 처음 상태로 |
| `POST /recommend` | 레퍼런스 사진 주소들 → 키워드 + 시안 N개(벽 색 + 바닥재 id) |
| `POST /rooms/{room_id}/generate` | 재료를 입힌 이미지 (`image_base64`). 슬라이더 값은 `options`로 |
| `DELETE /rooms/{room_id}` | 방 지우기 |

**영역 번호**: 1 = allWalls(벽 전체), 2 = floor(바닥), 3 = ceiling(천장), 0 = 없음(지우개)

**아직 지원 안 하는 것 (솔직하게)**
- 벽 여러 면 분리(`wall1~`)와 몰딩(`moldingTop/Bottom`): 요청에 와도 무시하고 `warnings`로 알려줌
- 벽지 무늬 사진: 사진의 평균 색으로만 칠함 (무늬를 입히는 기능은 아직 없음)
- 벽 재료는 `id`만으로는 알 수 없음 → `hex`나 `url`을 같이 보내야 함. 바닥재는 `/floors/sync`로 등록한 `id`면 됨
- `preview: true`는 결과를 작게 줄여서 보내 전송이 빨라질 뿐, 계산 자체가 줄지는 않음
- 처음 켤 때 모델을 내려받아서 몇 분 걸림. 서버에서 실제 속도는 아직 재보지 못했음

**한계**: 19번과 같음. 이 서버 코드도 **가짜 모델·가짜 AI 응답으로 테스트**했을 뿐, 실제 모델·실제 사진·실제 터널(cloudflared)로는 아직 실행 확인 전. 오류가 나면 메시지를 알려줘야 함

**준비**: Colab 🔑에 `OPENAI_API_KEY`, `AI_SERVER_KEY` 저장 + '노트북 액세스' 켜기. Colab 런타임은 GPU(T4 이상)로 설정. 개발용 임시 주소(Cloudflare Tunnel)는 서버를 다시 켜면 바뀜 → 바뀌면 백엔드에 알려야 함

In [ ]:
# 1단계: 라이브러리 설치
!pip install transformers torch pillow matplotlib opencv-python-headless openai ipywidgets fastapi uvicorn python-multipart requests --quiet

In [ ]:
# 2단계: 설정 (여기만 고치면 됨)

MODEL = "gpt-5.4-mini"    # 분석에 쓸 OpenAI 모델 (사진을 볼 수 있고 가벼운 모델. 계정에서 쓸 수 있는 이름으로 바꾸기)
N_PROPOSALS = 3           # 시안 개수 (벽 색 + 바닥재 조합). 2~5 권장
MAX_REFERENCES = 6        # 레퍼런스 사진 최대 장수 (1장도 가능)
SEND_MAX_SIDE = 512       # AI에 보내는 레퍼런스 사진의 긴 변 크기 (작을수록 빠름. 색·분위기 분석에는 512로 충분)

SEG_MODEL = ("oneformer", "shi-labs/oneformer_ade20k_swin_large")
# SEG_MODEL = ("mask2former", "facebook/mask2former-swin-large-ade-semantic")

# --- 바닥재 폴더 ---
FLOOR_FOLDER = "/content/drive/MyDrive/프로젝트/바닥재"   # 바닥재 사진을 모아둔 Drive 폴더
FLOOR_MIRROR_TILE = False  # True: 이음새가 있는 사진을 좌우·상하로 뒤집어 이어 붙여 이음새를 감춤 (무늬가 대칭으로 보일 수 있음)

# --- 바닥재 고르기 (CLIP) ---
CLIP_MODEL = "openai/clip-vit-base-patch32"   # 정확도를 더 올리고 싶으면 "openai/clip-vit-large-patch14" (느리고 무거움)
FLOOR_COLOR_WEIGHT = 4.0   # 대표 색이 비슷한 바닥재를 얼마나 우대할지. 0=색 무시, 클수록 색을 중요하게
FLOOR_FROM_REFERENCE = False   # True: 레퍼런스 사진의 바닥 부분을 잘라 바닥재 사진과 직접 비교하는 점수도 더함
FLOOR_REF_WEIGHT = 1.0     # 위 옵션을 켰을 때의 비중

# --- 바닥 깔기 (원근) ---
FLOOR_TILE = 0.5          # 바닥재 한 칸의 가로 크기 (화면 맨 아래 기준, 이미지 가로 대비). 작을수록 잘게 반복됨
HORIZON = None            # 수평선 높이 (이미지 높이 대비 0~1, 위가 0). None=자동. 줄이 이상하게 모이면 10단계 미리보기 보며 조절
FOV = 0.9                 # 카메라 화각 느낌 (클수록 원근이 약함, 0.7~1.3)
ROTATE = 0                # 바닥 무늬 회전 각도(도). 사진을 비스듬히 찍었으면 -30~30 사이로 조절

# --- 바닥 빛과 반사 ---
FLOOR_LIGHT = 1.0         # 원래 방의 빛(밝은 곳/그림자) 유지 정도. 0=평평, 1=원본 그대로
FLOOR_SPEC = 1.0          # 원래 사진의 반짝이는 하이라이트를 다시 얹는 정도
FLOOR_REFLECT = 0.0       # 거울 같은 반사. 0=없음, 0.3=은은한 광택, 0.7=반들반들
FLOOR_ROUGH = 0.3         # 반사의 흐릿함. 0=또렷, 1=매우 흐림
FLOOR_EXPOSURE = 1.0      # 바닥재 밝기 배율. 아주 하얀 바닥재에서 빛이 뭉치면 0.85쯤으로

In [ ]:
# 3단계: 영역 분할 (OneFormer를 한 번만 불러와서 계속 재사용 → 두 번째 사진부터는 몇 초)
import gc, io, time
import numpy as np
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_image_rgb(path, max_side=1024):
    """사진을 RGB로 읽고, 너무 크면 긴 변을 max_side로 줄여요."""
    image = Image.open(path).convert("RGB")
    w, h = image.size
    scale = max_side / max(w, h)
    if scale < 1:
        image = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return image

_seg_cache = {}
def get_segmenter():
    """분할 모델을 처음 한 번만 불러와요 (이후엔 그대로 재사용)."""
    kind, name = SEG_MODEL
    if kind not in _seg_cache:
        if kind == "oneformer":
            from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation
            processor = OneFormerProcessor.from_pretrained(name)
            model = OneFormerForUniversalSegmentation.from_pretrained(name).eval().to(DEVICE)
        elif kind == "mask2former":
            from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
            processor = AutoImageProcessor.from_pretrained(name)
            model = Mask2FormerForUniversalSegmentation.from_pretrained(name).eval().to(DEVICE)
        else:
            raise ValueError(f"알 수 없는 모델 종류: {kind}")
        _seg_cache[kind] = (processor, model)
    return _seg_cache[kind]

def find_class_id(id2label, name):
    for class_id, label in id2label.items():
        if label.split(",")[0].strip() == name:
            return int(class_id)
    raise KeyError(f"{name} 클래스를 찾을 수 없음")

def segment_room(image):
    """방 사진 → 벽/바닥/천장 마스크 (True/False 지도)."""
    processor, model = get_segmenter()
    kwargs = {"task_inputs": ["semantic"]} if SEG_MODEL[0] == "oneformer" else {}
    inputs = {k: v.to(DEVICE) for k, v in processor(images=image, return_tensors="pt", **kwargs).items()}
    with torch.no_grad():
        outputs = model(**inputs)
    seg = processor.post_process_semantic_segmentation(outputs, target_sizes=[image.size[::-1]])[0].cpu().numpy()
    id2label = model.config.id2label
    return {name: seg == find_class_id(id2label, name) for name in ["wall", "floor", "ceiling"]}

In [ ]:
# 4단계: 마스크 정리 + 벽 색 편집 + 바닥재 원근 + 바닥 빛/반사 함수
import cv2

def clean_mask(mask, min_area_ratio=0.002):
    """마스크의 지저분한 부분을 정리해요 (후처리).
    1) 작은 점/구멍 없애기(open/close)  2) 너무 작은 조각 제거  3) 작은 구멍 메우기
    """
    H, W = mask.shape
    mask_u8 = mask.astype(np.uint8)
    k = max(3, int(min(H, W) * 0.01) | 1)                       # 이미지 크기에 맞춘 홀수 커널
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)     # 튀어나온 점 제거
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)    # 작은 구멍 메우기

    min_area = H * W * min_area_ratio
    # 작은 조각 제거
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    for i in range(1, count):
        if stats[i, cv2.CC_STAT_AREA] < min_area:
            mask_u8[labels == i] = 0
    # 작은 구멍(배경 조각) 메우기
    holes = (1 - mask_u8).astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=4)
    for i in range(1, count):
        if stats[i, cv2.CC_STAT_AREA] < min_area:
            mask_u8[labels == i] = 1
    return mask_u8.astype(bool)


def hex_to_rgb(hex_code):
    hex_code = hex_code.lstrip("#")
    return tuple(int(hex_code[i:i + 2], 16) for i in (0, 2, 4))

def detail_shade(image, mask, blur=1.0, lo=0.45, hi=1.6):
    """원본의 밝기 무늬를 그대로 가져와요 (뭉개지 않고).
    벽에 턱이 있으면 턱 위/아래의 밝기 차이, 모서리 그림자, 창가 빛이 새 색 위에도 그대로 남음.
    마스크 안쪽의 평균 밝기를 1.0으로 맞춤."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32)
    if blur > 0:
        gray = cv2.GaussianBlur(gray, (0, 0), blur)          # 센서 잡음만 아주 살짝 줄임
    average = gray[mask].mean() if mask.any() else 128.0
    return np.clip(gray / (average + 1e-6), lo, hi)

def blend_region(image, mask, painted, edge=0.8):
    """칠한 결과(painted)를 마스크 영역에만 합쳐요. edge: 가장자리 부드러움(픽셀)."""
    alpha = mask.astype(np.float32)
    if edge > 0:
        alpha = cv2.GaussianBlur(alpha, (0, 0), edge)
    alpha = alpha[..., None]
    out = image.astype(np.float32) * (1 - alpha) + painted.astype(np.float32) * alpha
    return np.clip(out, 0, 255).astype(np.uint8)

def paint_fast(image, mask, source, detail=1.0, edge=0.8):
    """[벽용] 마스크 영역을 칠해요. source: "#RRGGBB" 색상 코드, 또는 이미지 크기와 같은 (높이, 너비, 3) 배열.
    detail: 원본 밝기 무늬 반영 정도 (0=완전 평평, 1=원본 그대로, 1.3=대비 더 강하게)
    edge: 가장자리 부드러움(픽셀). 0.8이면 계단만 없앨 정도로 또렷함"""
    if not mask.any():
        return image
    shade = 1.0 + detail * (detail_shade(image, mask) - 1.0)
    if isinstance(source, str):
        color = np.array(hex_to_rgb(source), np.float32)[None, None, :]
    else:
        color = source.astype(np.float32)
    painted = np.clip(color * shade[..., None], 0, 255)
    return blend_region(image, mask, painted, edge)

def load_texture(path, max_side=1024):
    """바닥재 사진을 읽어요 (너무 크면 줄임). 위에서 내려다본, 이음새 없이 반복되는 사진이 좋아요."""
    img = Image.open(path).convert("RGB")
    s = max_side / max(img.size)
    if s < 1:
        img = img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)
    return np.array(img)

def sample_marble(size=512, seed=0):
    """[테스트용] 가짜 대리석 무늬 (이음새 없이 반복됨). 진짜 사진이 생기면 load_texture로 바꿔 끼우면 돼요."""
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:size, 0:size] / size
    turb = np.zeros((size, size))
    for _ in range(6):
        kx, ky = rng.integers(1, 6, 2)
        turb += rng.uniform(0.15, 0.5) / (kx + ky) * np.sin(2 * np.pi * (kx * x + ky * y) + rng.uniform(0, 6.28))
    vein = np.exp(-np.abs(np.sin(2 * np.pi * (2 * x + 3 * y + 3 * turb))) * 4)
    vein2 = np.exp(-np.abs(np.sin(2 * np.pi * (3 * x - 2 * y + 4 * turb))) * 8) * 0.6
    cloud = 0.5 + 0.5 * np.sin(2 * np.pi * (x + 2 * turb)) * np.cos(2 * np.pi * (y - turb))
    base = np.array([238.0, 235.0, 228.0]) - cloud[..., None] * 14
    out = base - (vein + vein2)[..., None] * np.array([78.0, 80.0, 86.0])
    return np.clip(out, 0, 255).astype(np.uint8)


def estimate_horizon(mask, horizon=None):
    """수평선 높이(픽셀)를 정해요. horizon을 주면 그 비율(0~1)을, 없으면 사진 중간쯤을 쓰되 바닥 맨 윗줄보다는 위로 잡아요."""
    H = mask.shape[0]
    rows = np.where(mask.any(axis=1))[0]
    top = rows.min() if len(rows) else H // 2
    y_h = (horizon if horizon is not None else 0.5) * H
    return float(min(y_h, top - 0.02 * H))

def floor_perspective(shape, mask, texture, tile=0.5, horizon=None, fov=0.9, rotate=0.0):
    """바닥 영역에 원근을 맞춰 무늬를 깔아요 → (높이, 너비, 3) 배열.
    카메라가 바닥 위에서 수평으로 보고 있다고 가정: 수평선 위치(y_h)만 알면 화면 아래 픽셀이 바닥의 어느 지점인지 계산됨.
      깊이 Z = fx / (y - y_h),  좌우 X = (x - W/2) / (y - y_h)   (카메라 높이 = 1로 둠)
    멀리 갈수록 무늬가 작아지므로 7단계 크기별 축소본(미프맵)에서 골라 써서 깨알 같은 번쩍임을 막음."""
    H, W = shape
    y_h = estimate_horizon(mask, horizon)
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    d = np.maximum(ys - y_h, 1.0)
    fx = fov * W
    Z = fx / d
    X = (xs - W / 2.0) / d
    if rotate:
        a = np.deg2rad(rotate)
        X, Z = X * np.cos(a) - Z * np.sin(a), X * np.sin(a) + Z * np.cos(a)
    rows = np.where(mask.any(axis=1))[0]
    y_near = rows.max() if len(rows) else H - 1
    tile_world = tile * W / max(y_near - y_h, 1.0)             # 화면 맨 아래에서 무늬 한 칸이 이미지 가로의 tile 비율이 되도록
    th, tw = texture.shape[:2]
    u = X / tile_world * tw
    v = Z / tile_world * tw                                    # 무늬의 가로세로 비율 유지 (세로 한 칸 = 가로 한 칸 × th/tw)
    # 화면 한 픽셀이 무늬 몇 픽셀에 해당하는지 → 어느 크기의 축소본을 쓸지
    foot = np.maximum(tw / (tile_world * d), fx * tw / (tile_world * d * d))
    level = np.clip(np.log2(np.maximum(foot, 1.0)), 0, 6)
    out = np.zeros((H, W, 3), np.float32)
    for k in range(7):
        w = np.clip(1 - np.abs(level - k), 0, 1)
        if not (w > 0).any():
            continue
        s = 2 ** k
        tex_k = texture if k == 0 else cv2.resize(texture, (max(4, tw // s), max(4, th // s)), interpolation=cv2.INTER_AREA)
        hk, wk = tex_k.shape[:2]
        mx = ((u * wk / tw) % wk).astype(np.float32)
        my = ((v * hk / th) % hk).astype(np.float32)
        sample = cv2.remap(tex_k, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)
        out += sample.astype(np.float32) * w[..., None]
    mean_color = texture.reshape(-1, 3).mean(axis=0)
    fade = np.clip((np.log2(np.maximum(foot, 1.0)) - 6.0) / 1.5, 0, 1)[..., None]   # 아주 먼 곳은 무늬 평균색으로 자연스럽게
    out = out * (1 - fade) + mean_color[None, None, :] * fade
    return np.clip(out, 0, 255).astype(np.uint8)

def _lin(x):
    """화면 밝기값(0~255)을 실제 빛의 양(0~1)으로 바꿔요. 빛은 이 단위에서 곱해야 자연스러움."""
    return (np.asarray(x, np.float32) / 255.0) ** 2.2

def _gamma(x):
    return np.clip(x, 0, 1) ** (1 / 2.2) * 255.0

def floor_lighting(image, mask, base, keep=1.0, spec=1.0, exposure=1.0, sigma_ratio=0.012):
    """바닥재(base)에 원래 방의 빛을 입혀요. 빛을 두 가지로 나눠서 따로 다룸.
      1) 퍼진 빛(밝은 곳/그림자): 원본 바닥 밝기를 살짝 흐려서(원래 바닥 무늬는 지우고 빛만 남김) 새 바닥재에 곱함
      2) 반짝임(하이라이트): 주변보다 확 밝은 부분만 따로 뽑아서 흰빛으로 다시 얹음
    밝은 바닥재일 때 빛이 하얗게 날아가던 문제를 막으려고, 밝아지는 쪽은 \'남은 여유(1에서 모자란 만큼)\'를 넘지 않게 부드럽게 눌러줌.
    exposure: 바닥재 자체의 밝기 배율. 아주 하얀 바닥재라면 0.8~0.9로 낮춰야 빛의 모양이 보일 여유가 생김."""
    H, W = mask.shape
    lin = _lin(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY))
    m = mask.astype(np.float32)
    sigma = max(3.0, sigma_ratio * max(H, W))
    low = cv2.GaussianBlur(lin * m, (0, 0), sigma) / (cv2.GaussianBlur(m, (0, 0), sigma) + 1e-6)   # 마스크 안쪽만 보는 흐린 밝기
    average = (lin * m).sum() / (m.sum() + 1e-6)
    ratio = np.clip(low / (average + 1e-6), 0.15, 4.0)
    diffuse = ratio ** keep                                          # keep=0이면 1(빛 없음), 1이면 원본 그대로
    glare = np.clip(lin - low * 1.08, 0, None) / (average + 1e-6)    # 주변 빛보다 8% 넘게 밝은 부분
    b = _lin(base) * exposure
    r = diffuse[..., None]
    room_left = np.maximum(0.97 - b, 0.03)                              # 더 밝아질 수 있는 여유
    brighter = b + room_left * (1 - np.exp(-(r - 1) * b / room_left))    # 밝아질 땐 여유 안에서 부드럽게 (기울기는 곱셈과 같게 시작)
    out = np.where(r >= 1, brighter, b * r)                              # 어두워질 땐 그냥 곱함 (그림자)
    out = out + spec * 0.5 * np.clip(glare, 0, 2)[..., None]
    return _gamma(out)

def floor_reflection(scene, mask, y_h, y_near, rough=0.3):
    """바닥에 비친 모습(거울상)을 만들어요 → (반사 이미지, 세기 지도).
    바닥 윗가장자리(벽과 만나는 선)를 거울면으로 보고, 선 아래 픽셀은 선 위의 같은 거리만큼 위쪽 장면을 비춤.
    멀리 있는 바닥일수록 더 잘 비치도록(프레넬 느낌) 세기를 줌. 가구는 벽 기준으로 대충 비추므로 근사임."""
    H, W = mask.shape
    cols = mask.any(axis=0)
    base_y = np.where(cols, mask.argmax(axis=0), 0).astype(np.float32)
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    map_y = 2.0 * base_y[None, :] - ys
    valid = ((map_y >= 0) & (map_y < ys)).astype(np.float32)
    map_yc = np.clip(map_y, 0, H - 1).astype(np.float32)
    sharp = cv2.remap(scene, xs, map_yc, cv2.INTER_LINEAR).astype(np.float32)
    blurred = cv2.GaussianBlur(sharp, (0, 0), 0.025 * max(H, W))
    refl = sharp * (1 - rough) + blurred * rough
    src_is_floor = cv2.remap(mask.astype(np.float32), xs, map_yc, cv2.INTER_LINEAR)
    weight = valid * (1 - np.clip(src_is_floor * 1.5, 0, 1)) * np.clip(map_y / (0.08 * H), 0, 1)   # 바닥이 바닥을 비추거나 사진 맨 위는 약하게
    t = np.clip((ys - y_h) / max(y_near - y_h, 1.0), 0, 1)           # 0=먼 곳, 1=가까운 곳
    fresnel = 0.25 + 0.75 * (1 - t) ** 2
    return refl, weight * fresnel

def light_floor(room, scene, base, keep=None, spec=None, reflect=None, rough=None, exposure=None):
    """바닥재(base: 색상 코드 또는 배열)에 빛(퍼진 빛 + 하이라이트)과 반사를 적용한 바닥 이미지를 만들어요."""
    keep = FLOOR_LIGHT if keep is None else keep
    spec = FLOOR_SPEC if spec is None else spec
    reflect = FLOOR_REFLECT if reflect is None else reflect
    rough = FLOOR_ROUGH if rough is None else rough
    exposure = FLOOR_EXPOSURE if exposure is None else exposure
    mask = room["masks"]["floor"]
    H, W = mask.shape
    if isinstance(base, str):
        base = np.full((H, W, 3), hex_to_rgb(base), np.uint8)
    lit = floor_lighting(room["image"], mask, base, keep, spec, exposure)
    if reflect > 0:
        rows = np.where(mask.any(axis=1))[0]
        y_h = estimate_horizon(mask, HORIZON)
        refl, strength = floor_reflection(scene, mask, y_h, rows.max(), rough)
        w = np.clip(reflect * strength, 0, 1)[..., None]
        lit = lit * (1 - w) + refl * w
    return lit

def recolor(room, wall_hex=None, floor=None, detail=1.0, edge=0.8, floor_laid=None,
            keep=None, spec=None, reflect=None, rough=None, exposure=None):
    """준비된 방에 벽 색과 바닥을 바로 입혀요. 분할은 다시 안 함.
    floor: "#RRGGBB" 색상 코드, 또는 바닥재 사진 배열(원근을 맞춰 깔아줌)
    floor_laid: 이미 원근으로 깔아 둔 바닥 무늬(같은 바닥재로 여러 장 만들 때 한 번만 계산해서 재사용)
    keep / spec / reflect / rough / exposure: 바닥의 빛 유지 / 하이라이트 / 반사 / 반사 흐림 / 바닥재 밝기 (안 쓰면 2단계의 FLOOR_* 값)"""
    out = np.array(room["image"])
    if wall_hex:
        out = paint_fast(out, room["masks"]["wall"], wall_hex, detail, edge)
    if floor_laid is None and floor is not None and not isinstance(floor, str):
        floor_laid = floor_perspective(out.shape[:2], room["masks"]["floor"], floor, FLOOR_TILE, HORIZON, FOV, ROTATE)
    base = floor_laid if floor_laid is not None else floor
    if base is not None and room["masks"]["floor"].any():
        painted = light_floor(room, out, base, keep, spec, reflect, rough, exposure)       # 반사에는 새 벽 색이 비침
        out = blend_region(out, room["masks"]["floor"], painted, edge)
    return out

In [ ]:
# 5단계: 바닥재 목록 만들기 + CLIP 벡터 (바닥재 파일 이름은 아무거나 OK)
import os, io

FLOOR_EXTS = (".jpg", ".jpeg", ".png", ".webp")
_floor_cache = {}       # 바닥재 id → 읽어 둔 사진 (한 번만 읽음)

# ---- CLIP: 사진/글을 같은 공간의 숫자(벡터)로 바꿔서 서로 얼마나 닮았는지 비교 ----
_clip_cache = {}
def get_clip():
    """CLIP 모델을 처음 한 번만 불러와요."""
    if "m" not in _clip_cache:
        from transformers import CLIPModel, CLIPProcessor
        print(f"  CLIP 불러오는 중: {CLIP_MODEL}")
        _clip_cache["m"] = (CLIPProcessor.from_pretrained(CLIP_MODEL), CLIPModel.from_pretrained(CLIP_MODEL).eval().to(DEVICE))
    return _clip_cache["m"]

def _to_vectors(out):
    """CLIP 출력을 길이 1로 맞춘 numpy 벡터로 (transformers 버전에 따라 출력 모양이 달라서 둘 다 처리)."""
    if not torch.is_tensor(out):
        out = out.pooler_output if getattr(out, "pooler_output", None) is not None else out[0]
    v = out.detach().cpu().numpy().astype(np.float32)
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)

def embed_images(images, batch=16):
    """PIL 사진 리스트 → (장수, 차원) 벡터"""
    processor, model = get_clip()
    outs = []
    for i in range(0, len(images), batch):
        inputs = processor(images=[im.convert("RGB") for im in images[i:i + batch]], return_tensors="pt")
        with torch.no_grad():
            outs.append(_to_vectors(model.get_image_features(pixel_values=inputs["pixel_values"].to(DEVICE))))
    return np.concatenate(outs)

def embed_texts(texts):
    """글 리스트 → (개수, 차원) 벡터"""
    processor, model = get_clip()
    inputs = processor(text=list(texts), return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = model.get_text_features(input_ids=inputs["input_ids"].to(DEVICE), attention_mask=inputs["attention_mask"].to(DEVICE))
    return _to_vectors(out)

def avg_hex(path_or_array):
    """바닥재 사진의 평균 색을 #RRGGBB로."""
    img = Image.fromarray(path_or_array) if isinstance(path_or_array, np.ndarray) else Image.open(path_or_array).convert("RGB")
    small = np.array(img.resize((32, 32), Image.BILINEAR)).reshape(-1, 3).mean(axis=0).astype(int)
    return "#{:02X}{:02X}{:02X}".format(*small)

def _read_floor_raw(info):
    """바닥재 한 장을 배열로 읽어요 (path 또는 url)."""
    if info.get("path"):
        return load_texture(info["path"])
    import requests
    r = requests.get(info["url"], timeout=20); r.raise_for_status()
    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    s = 1024 / max(img.size)
    if s < 1:
        img = img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)
    return np.array(img)

def make_floor_vector(info):
    """[백엔드용] 바닥재 한 장({"path": ...} 또는 {"url": ...})의 CLIP 벡터를 리스트로 돌려줘요. 바닥재를 등록할 때 한 번 계산해서 DB에 저장해 두면 돼요."""
    return embed_images([Image.fromarray(_read_floor_raw(info))])[0].tolist()

def _finish_library(entries):
    """읽어 둔 바닥재들에 평균 색과 CLIP 벡터를 채워요. 벡터가 이미 있는 건 그대로 쓰고, 없는 것만 한꺼번에 계산."""
    need = [fid for fid, e in entries.items() if e.get("vec") is None]
    if need:
        print(f"  바닥재 {len(need)}개를 CLIP으로 벡터화 중...")
        for fid, v in zip(need, embed_images([Image.fromarray(entries[fid]["raw"]) for fid in need])):
            entries[fid]["vec"] = v
    library = {}
    for fid, e in entries.items():
        v = np.asarray(e["vec"], np.float32)
        library[fid] = {"path": e.get("path"), "url": e.get("url"), "avg_hex": avg_hex(e["raw"]), "vec": v / (np.linalg.norm(v) + 1e-9)}
    return library

def set_floor_library(items):
    """[백엔드 연결용] 바닥재 목록을 직접 넣어요.
    items: [{"id": 아무 이름이나 번호, "url": "https://..." 또는 "path": "/파일/경로", "vec": [...] (선택)}]
    vec(CLIP 벡터)을 같이 주면 계산을 건너뛰어 빨라요 (make_floor_vector로 미리 만들어 DB에 저장해 둔 값)."""
    global FLOOR_LIBRARY
    entries = {}
    for it in items:
        fid = str(it["id"])
        info = {"path": it.get("path"), "url": it.get("url")}
        if not (info["path"] or info["url"]):
            print(f"  바닥재 '{fid}'에 path나 url이 없어 건너뜀"); continue
        try:
            _floor_cache.pop(fid, None)
            entries[fid] = {**info, "raw": _read_floor_raw(info), "vec": it.get("vec")}
        except Exception as e:
            print(f"  바닥재 '{fid}'를 읽지 못해 건너뜀: {type(e).__name__}: {str(e)[:80]}")
    if not entries:
        raise RuntimeError("쓸 수 있는 바닥재가 하나도 없어요")
    FLOOR_LIBRARY = _finish_library(entries)
    print(f"바닥재 {len(FLOOR_LIBRARY)}개 등록")
    return FLOOR_LIBRARY

def load_floor_library(folder=None):
    """바닥재 폴더를 읽어 {id(파일 이름): {"path", "avg_hex", "vec"}} 목록을 만들어요.
    폴더가 없거나 비어 있으면 가짜 대리석 하나로 대신해서 구조만 테스트할 수 있게 함."""
    folder = folder or FLOOR_FOLDER
    if not os.path.isdir(folder):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print(f"  (Drive 연결 안 됨: {type(e).__name__})")
    entries = {}
    if os.path.isdir(folder):
        for name in sorted(os.listdir(folder)):
            stem, ext = os.path.splitext(name)
            if ext.lower() in FLOOR_EXTS:
                path = os.path.join(folder, name)
                try:
                    entries[stem] = {"path": path, "raw": load_texture(path), "vec": None}
                except Exception as e:
                    print(f"  바닥재 '{name}'을 읽지 못해 건너뜀: {type(e).__name__}")
    if not entries:
        print(f"⚠️ 바닥재 폴더가 비어 있거나 없음: {folder}")
        print("   → 가짜 대리석(sample_marble) 하나로 테스트합니다. 진짜 바닥재를 넣으면 자동으로 사용돼요.")
        _floor_cache["sample_marble"] = sample_marble()
        entries["sample_marble"] = {"path": None, "raw": _floor_cache["sample_marble"], "vec": None}
    library = _finish_library(entries)
    print(f"바닥재 {len(library)}개: " + ", ".join(library))
    return library

def get_floor_texture(floor_id, library):
    """바닥재 id로 사진(배열)을 가져와요 (처음 한 번만 읽음)."""
    if floor_id not in _floor_cache:
        tex = _read_floor_raw(library[floor_id])
        if FLOOR_MIRROR_TILE:
            top = np.concatenate([tex, tex[:, ::-1]], axis=1)
            tex = np.concatenate([top, top[::-1]], axis=0)
        _floor_cache[floor_id] = tex
    return _floor_cache[floor_id]

FLOOR_LIBRARY = load_floor_library()

In [ ]:
# 6단계: 레퍼런스 분석 함수 (OpenAI)
import os, json, re, time, base64

HEX_PATTERN = re.compile(r"^#[0-9A-Fa-f]{6}$")

def get_key(name, tries=3):
    """Colab 보안 비밀 → 환경변수 순서로 키를 찾아요. 없으면 None (그 API는 건너뜀).
    보안 비밀을 읽다가 일시적으로 시간 초과(TimeoutException)가 나면 몇 번 더 시도해요."""
    for attempt in range(tries):
        try:
            from google.colab import userdata
            key = userdata.get(name)
            if isinstance(key, str) and key.strip():
                return key.strip()
            break                                   # 읽기는 됐는데 값이 비어 있음 → 다시 해도 소용없음
        except Exception as e:
            transient = "timeout" in type(e).__name__.lower() or "timeout" in str(e).lower()
            print(f"  ({name}) Colab 보안 비밀을 못 불러옴 ({attempt + 1}/{tries}): {type(e).__name__}")
            if not transient:
                print("   → 🔑에 이름이 정확한지, '노트북 액세스'가 켜져 있는지 확인")
                break
            time.sleep(2)
    key = os.environ.get(name)
    if isinstance(key, str) and key.strip():
        return key.strip()
    print(f"  ({name}) 키를 찾지 못해 이 API는 건너뜀")
    return None

_client = None
def get_client():
    """OpenAI 클라이언트를 한 번만 만들어 재사용. 키가 없으면 None. (키를 못 찾은 경우는 기억하지 않아서 다시 부르면 또 찾아봐요)"""
    global _client
    if _client is None:
        key = get_key("OPENAI_API_KEY")
        if not key:
            return None
        from openai import OpenAI
        _client = OpenAI(api_key=key)
    return _client


ANALYSIS_PROMPT = """너는 실내 인테리어 컬러·소재 코디네이터야.
첨부한 레퍼런스 사진들(1장일 수도 있어)은 한 사용자가 좋아하는 방이야. 이 사진들을 분석해서
(1) 사용자의 인테리어 취향 키워드를 뽑고,
(2) 사용자 방에 적용할 시안을 __N__가지 제안해줘. 시안 하나는 '벽 색 1개 + 바닥재 느낌 1개'야.

규칙:
- 키워드는 사진들의 공통된 분위기·색감·소재·질감에서 뽑아. keywords_ko는 한국어 5~8개, keywords_en은 같은 뜻의 영어 5~8개(소문자).
- 시안 __N__가지는 모두 레퍼런스의 취향과 어울려야 하지만 서로 확실히 달라야 해. 같은 분위기 안에서 밝기와 깊이를 다르게 하고(밝은 / 중간 / 어두운 톤 등), 바닥재 재질이나 톤도 서로 다르게 해줘.
- 벽 색과 바닥재는 서로 자연스럽게 어울려야 해. 벽 색은 가구가 아니라 벽의 색이야.
- 바닥재는 레퍼런스 사진 속 바닥의 재질·톤과 비슷하게 하되, 문장이 아니라 영어 단어 2~4개로 말해줘. 재질, 톤, 질감 순서로 (예: ["light oak", "wood plank", "matte"], ["white marble", "polished"]). floor_hex는 그 바닥재의 대표 색.
- hex는 반드시 #RRGGBB 형식. color_name은 한국어로 짧게. name은 시안 이름(한국어, 짧게). tone은 light | mid | dark 중 하나.
- reason은 한국어 2문장 이내로 키워드와 선택의 이유.

아래 JSON 형식으로만 답해:
{
  "keywords_ko": ["", ""],
  "keywords_en": ["", ""],
  "style": "스타일 이름(영어)",
  "warmth": "warm | cool | neutral",
  "softness": "soft | hard | neutral",
  "proposals": [
    {"name": "", "tone": "light | mid | dark",
     "wall": {"color_name": "", "hex": ""},
     "floor_keywords_en": ["", ""],
     "floor_hex": ""}
  ],
  "reason": ""
}"""

def build_prompt():
    return ANALYSIS_PROMPT.replace("__N__", str(N_PROPOSALS))

def validate_analysis(data):
    """AI 응답이 약속한 형식인지 검사하고 값을 정리해요. 이상하면 ValueError → 다시 시도."""
    if not isinstance(data, dict):
        raise ValueError("JSON 객체(dict)가 아님")
    for key in ["keywords_ko", "keywords_en"]:
        items = data.get(key)
        if not isinstance(items, list) or len([k for k in items if isinstance(k, str) and k.strip()]) < 3:
            raise ValueError(f"'{key}'는 문자열 3개 이상의 리스트여야 함")
        data[key] = [k.strip() for k in items if isinstance(k, str) and k.strip()]
    data["keywords_en"] = [k.lower() for k in data["keywords_en"]]
    proposals = data.get("proposals")
    if not isinstance(proposals, list) or len(proposals) < 1:
        raise ValueError("'proposals'는 시안 1개 이상의 리스트여야 함")
    clean = []
    for i, p in enumerate(proposals[:N_PROPOSALS]):
        wall = p.get("wall") if isinstance(p, dict) else None
        if not isinstance(wall, dict) or not wall.get("color_name") or not HEX_PATTERN.match(str(wall.get("hex", ""))):
            raise ValueError(f"proposals[{i}].wall에 color_name과 #RRGGBB 형식의 hex가 필요함")
        words = p.get("floor_keywords_en")
        if isinstance(words, str):
            words = [w for w in words.split(",")]
        if not isinstance(words, list):
            raise ValueError(f"proposals[{i}].floor_keywords_en은 영어 단어 리스트여야 함")
        words = [w.strip().lower() for w in words if isinstance(w, str) and w.strip()][:6]
        if not words:
            raise ValueError(f"proposals[{i}].floor_keywords_en이 비어 있음")
        if not HEX_PATTERN.match(str(p.get("floor_hex", ""))):
            raise ValueError(f"proposals[{i}].floor_hex가 #RRGGBB 형식이 아님")
        tone = str(p.get("tone", "")).strip().lower()
        clean.append({"name": str(p.get("name") or f"시안 {i + 1}"), "tone": tone if tone in ["light", "mid", "dark"] else "mid",
                      "wall": wall, "floor_keywords_en": words, "floor_hex": p["floor_hex"]})
    data["proposals"] = clean
    for key, allowed in [("warmth", ["warm", "cool", "neutral"]), ("softness", ["soft", "hard", "neutral"])]:
        value = str(data.get(key, "")).strip().lower()
        data[key] = value if value in allowed else "neutral"
    data.setdefault("style", "")
    if not data.get("reason"):
        raise ValueError("'reason' 항목이 없음")
    return data

def load_image_bytes(path, max_side=1024):
    """AI에 보내려고 JPEG 바이트로 변환 (확장자가 뭐든 같은 형식으로 통일)."""
    image = load_image_rgb(path, max_side)
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=90)
    return buffer.getvalue()

def _call_openai(reference_paths):
    content = [{"type": "text", "text": build_prompt()}]
    for path in reference_paths:
        b64 = base64.b64encode(load_image_bytes(path, SEND_MAX_SIDE)).decode()
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})
    response = get_client().chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": content}],
        response_format={"type": "json_object"})        # temperature는 모델에 따라 못 쓰는 경우가 있어 지정하지 않음
    return response.choices[0].message.content


# 재시도해도 해결되지 않는 오류 (단어들, 안내 문구). 위에 있는 것부터 확인
FATAL_ERRORS = [
    (["insufficient_quota", "no credits", "billing", "exceeded your current quota"], "API 크레딧/결제 한도 문제. OpenAI 결제 페이지에서 크레딧을 충전해야 함 (ChatGPT Plus 같은 구독과는 별개)"),
    (["invalid_api_key", "incorrect api key", "401"], "API 키가 올바르지 않음. 🔑에 저장한 키를 다시 확인"),
    (["permission", "403"], "이 키로는 이 모델을 쓸 권한이 없음"),
    (["model_not_found", "404", "not found"], "모델 이름이 없거나 쓸 수 없음. 2단계의 MODEL 확인"),
]

def analyze_reference(reference_paths, max_retry=3):
    """[AI 분석] 레퍼런스 사진들 → 키워드 + 시안(벽 색 + 바닥재 이름).
    - 형식이 틀리거나 없는 바닥재를 골랐으면 다시 시도, 서버 혼잡(429/500/503 등)이면 점점 더 오래 기다렸다가 다시 시도
    - 끝까지 실패하면 RuntimeError
    반환: (분석 결과 dict, 걸린 시간(초))"""
    if not reference_paths:
        raise RuntimeError("레퍼런스 사진이 없어요 (1장 이상 필요)")
    reference_paths = list(reference_paths)[:MAX_REFERENCES]
    if get_client() is None:
        raise RuntimeError("OpenAI API 키가 없어요. Colab 🔑에 OPENAI_API_KEY를 저장하고 '노트북 액세스'를 켜주세요")
    started, last_error = time.time(), None
    for attempt in range(max_retry + 1):
        try:
            text = _call_openai(reference_paths)
            data = validate_analysis(json.loads(text))
            return data, time.time() - started
        except Exception as e:
            last_error = e
            msg = str(e).lower()
            for words, hint in FATAL_ERRORS:      # 다시 시도해도 소용없는 오류는 바로 멈추고 이유를 알려줘요
                if any(w in msg for w in words):
                    raise RuntimeError(f"OpenAI 사용 불가 - {hint}")
            print(f"  (재시도 {attempt + 1}/{max_retry + 1}) {type(e).__name__}: {str(e)[:150]}")
            busy = any(w in msg for w in ["429", "500", "502", "503", "unavailable", "rate", "overloaded", "timeout"])
            if busy and attempt < max_retry:
                time.sleep(min(2 ** (attempt + 1), 20))
    raise RuntimeError(f"OpenAI 응답 실패: {last_error}")

In [ ]:
# 7단계: 실행 함수 (방 준비 1번 + 분석 1번 → CLIP으로 바닥재 고르기 → 시안별 합성)

def prepare_room(room_path):
    """방 사진을 읽고 벽/바닥을 한 번만 분할해 둬요. (이 결과로 색·바닥재를 몇 번이든 바로 바꿔 입힐 수 있음)"""
    started = time.time()
    image = load_image_rgb(room_path)
    masks = {name: clean_mask(m) for name, m in segment_room(image).items()}
    print(f"[영역 분할] {time.time() - started:.1f}초 | 벽 {masks['wall'].mean():.0%}, 바닥 {masks['floor'].mean():.0%}")
    return {"image": np.array(image), "masks": masks}

FLOOR_QUERY = "a top-down photo of {} flooring"

def reference_floor_vector(reference_paths):
    """레퍼런스 사진들에서 바닥 부분만 잘라 CLIP 벡터의 평균을 만들어요. 바닥이 안 보이는 사진은 건너뜀. 하나도 없으면 None."""
    vecs = []
    for path in reference_paths:
        try:
            image = load_image_rgb(path)
            mask = clean_mask(segment_room(image)["floor"])
            if mask.mean() < 0.02:
                continue
            arr = np.array(image).copy()
            arr[~mask] = arr[mask].mean(axis=0)                 # 바닥이 아닌 곳은 평균색으로 덮어서 바닥만 보이게
            ys, xs = np.where(mask)
            vecs.append(embed_images([Image.fromarray(arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1])])[0])
        except Exception as e:
            print(f"  (레퍼런스 바닥 분석 건너뜀: {type(e).__name__}: {str(e)[:60]})")
    if not vecs:
        return None
    v = np.mean(vecs, axis=0)
    return v / (np.linalg.norm(v) + 1e-9)

def pick_floors(proposals, ref_vec=None, top_k=3):
    """시안마다 바닥재를 하나씩 골라 p["floor_id"]에 넣어요 (p["floor_candidates"]에는 후보 상위 top_k).
    점수 = CLIP 단어 유사도(바닥재들 사이에서 표준화) + (선택) 레퍼런스 바닥과의 유사도 - 색 차이 × FLOOR_COLOR_WEIGHT
    시안끼리 같은 바닥재가 안 겹치도록 점수 높은 짝부터 확정함 (바닥재가 부족하면 겹침 허용)."""
    ids = list(FLOOR_LIBRARY)
    V = np.stack([FLOOR_LIBRARY[i]["vec"] for i in ids])
    T = embed_texts([FLOOR_QUERY.format(", ".join(p["floor_keywords_en"])) for p in proposals])
    sim = T @ V.T
    z = (sim - sim.mean(axis=1, keepdims=True)) / (sim.std(axis=1, keepdims=True) + 1e-6)
    if ref_vec is not None:
        rs = V @ ref_vec
        z = z + FLOOR_REF_WEIGHT * ((rs - rs.mean()) / (rs.std() + 1e-6))[None, :]
    avg = np.array([hex_to_rgb(FLOOR_LIBRARY[i]["avg_hex"]) for i in ids], np.float32)
    want = np.array([hex_to_rgb(p["floor_hex"]) for p in proposals], np.float32)
    dist = np.linalg.norm(want[:, None, :] - avg[None, :, :], axis=2) / 441.7      # 0(같은 색)~1
    score = z - FLOOR_COLOR_WEIGHT * dist
    chosen, used = {}, set()
    for flat in np.argsort(-score, axis=None):
        i, j = divmod(int(flat), len(ids))
        if i in chosen or j in used:
            continue
        chosen[i] = j; used.add(j)
        if len(chosen) == len(proposals):
            break
    for i in range(len(proposals)):
        if i not in chosen:
            chosen[i] = int(score[i].argmax())
    for i, p in enumerate(proposals):
        p["floor_id"] = ids[chosen[i]]
        p["floor_candidates"] = [ids[j] for j in np.argsort(-score[i])[:top_k]]
    return proposals

def make_design(room, reference_paths, detail=1.0):
    """레퍼런스 → 키워드 + 시안 N개(벽 색 + 바닥재 느낌) → CLIP으로 바닥재 고르기 → 시안마다 합성.
    반환: {"analysis", "seconds", "images", "floor_ids"} (images[i]가 analysis["proposals"][i]에 해당)"""
    analysis, seconds = analyze_reference(reference_paths)
    started = time.time()
    ref_vec = reference_floor_vector(list(reference_paths)[:MAX_REFERENCES]) if FLOOR_FROM_REFERENCE else None
    pick_floors(analysis["proposals"], ref_vec)
    laid_cache = {}                                      # 같은 바닥재는 원근 계산을 한 번만
    images, floor_ids = [], []
    for p in analysis["proposals"]:
        fid = p["floor_id"]
        if fid not in laid_cache:
            laid_cache[fid] = floor_perspective(room["image"].shape[:2], room["masks"]["floor"], get_floor_texture(fid, FLOOR_LIBRARY),
                                                FLOOR_TILE, HORIZON, FOV, ROTATE)
        images.append(recolor(room, p["wall"]["hex"], None, detail, floor_laid=laid_cache[fid]))
        floor_ids.append(fid)
    print(f"[OpenAI] 분석 {seconds:.1f}초 + 바닥재 고르기·합성 {len(images)}장 {time.time() - started:.1f}초")
    print("키워드:", ", ".join(analysis["keywords_ko"]))
    for i, p in enumerate(analysis["proposals"], 1):
        print(f"  {i}. {p['name']} ({p['tone']}) | 벽 {p['wall']['color_name']} {p['wall']['hex']} / 바닥재 {p['floor_id']}"
              f"  ← '{', '.join(p['floor_keywords_en'])}' (후보: {', '.join(p['floor_candidates'])})")
    return {"analysis": analysis, "seconds": seconds, "images": images, "floor_ids": floor_ids}

In [ ]:
# 8단계: AI 서버 코드 (FastAPI) — 백엔드가 부르는 창구
import uuid, base64, threading, collections, contextlib
import requests as _requests
from typing import List, Optional, Dict, Any
from PIL import ImageOps
from fastapi import FastAPI, Header, Depends, UploadFile, File, Form
from fastapi.responses import JSONResponse, Response
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel

# ---- 서버 설정 ----
AI_SERVER_KEY = get_key("AI_SERVER_KEY")      # 백엔드와 약속한 비밀 값 (요청 헤더 X-AI-Key)
if not AI_SERVER_KEY:
    print("⚠️ AI_SERVER_KEY가 없어 인증 없이 열립니다 (개발용). 백엔드와 정한 비밀 값을 Colab 🔑에 AI_SERVER_KEY로 저장하세요")
SERVER_MAX_SIDE = 1280      # 방 사진의 긴 변 최대 크기. 이보다 크면 줄여서 처리하고 결과도 이 크기 (클수록 선명하지만 느림)
PREVIEW_SIDE = 512          # generate의 preview=true일 때 결과의 긴 변
MAX_ROOMS = 30              # 서버가 기억하는 방 개수 (넘으면 오래된 것부터 지움)

REGIONS = {"allWalls": 1, "floor": 2, "ceiling": 3}           # 영역 이름 → 번호 (0은 지우개/없음)
REGION_NAMES = {v: k for k, v in REGIONS.items()}
REGION_COLORS = {1: (66, 133, 244), 2: (234, 67, 53), 3: (52, 168, 83)}

LOCK = threading.RLock()                 # 모델과 설정값을 한 번에 한 요청씩만 쓰게 함
ROOMS = collections.OrderedDict()        # room_id → 방 정보
EXTRA_FLOORS = collections.OrderedDict() # 목록에 없이 url로 바로 온 바닥재 사진 (최근 20개)

class AIError(Exception):
    def __init__(self, status, code, message, details=None):
        self.status, self.code, self.message, self.details = status, code, message, details

def _err(status, code, message, details=None):
    return JSONResponse(status_code=status, content={"error": {"code": code, "message": message, "details": details}})

app = FastAPI(title="Imagine AI Server", version="1.0",
              description="방 사진 영역 분할 + 벽 색/바닥재 합성 + 레퍼런스 취향 분석. 인증: 헤더 X-AI-Key (/health 제외). 에러: {error:{code,message,details}}")

@app.exception_handler(AIError)
async def _ai_error_handler(request, exc):
    return _err(exc.status, exc.code, exc.message, exc.details)

@app.exception_handler(RequestValidationError)
async def _validation_handler(request, exc):
    return _err(422, "invalid_request", "요청 형식이 올바르지 않아요", [{"loc": [str(x) for x in e["loc"]], "msg": e["msg"]} for e in exc.errors()])

def check_key(x_ai_key: Optional[str] = Header(None)):
    if AI_SERVER_KEY and x_ai_key != AI_SERVER_KEY:
        raise AIError(401, "unauthorized", "X-AI-Key 헤더가 없거나 올바르지 않아요")

# ---- 도우미 ----
@contextlib.contextmanager
def override_globals(**changes):
    """요청 한 건 동안만 설정값(2단계)을 바꿔요. None이면 그대로 둠. LOCK 안에서 써야 함."""
    g = globals()
    changes = {k: v for k, v in changes.items() if v is not None}
    old = {k: g[k] for k in changes}
    g.update(changes)
    try:
        yield
    finally:
        g.update(old)

def _download(url, what="이미지"):
    try:
        r = _requests.get(url, timeout=20); r.raise_for_status()
        return r.content
    except Exception as e:
        raise AIError(400, "download_failed", f"{what}을 내려받지 못했어요 ({type(e).__name__})", {"url": url})

def _pil_from_bytes(data, what="이미지"):
    try:
        return ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")      # 휴대폰 사진 회전 반영
    except Exception:
        raise AIError(400, "invalid_image", f"{what}을 열 수 없어요 (jpg, png 같은 사진 파일이어야 해요)")

def _resize_max(img, max_side):
    s = max_side / max(img.size)
    return img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), Image.LANCZOS) if s < 1 else img

def _masks_from_label(label):
    return {"wall": label == 1, "floor": label == 2, "ceiling": label == 3}

def _regions_info(room):
    return [{"region_id": i, "name": n, "area_ratio": round(float((room["label"] == i).mean()), 4)} for i, n in REGION_NAMES.items()]

def _get_room(room_id):
    with LOCK:
        room = ROOMS.get(room_id)
        if room is None:
            raise AIError(404, "room_not_found", "방 정보를 찾을 수 없어요. 서버가 다시 시작됐다면 POST /rooms로 사진을 다시 올려 주세요", {"room_id": room_id})
        ROOMS.move_to_end(room_id)
        return room

def _room_summary(room_id, room):
    H, W = room["label"].shape
    ow, oh = room["orig_size"]
    return {"room_id": room_id, "width": ow, "height": oh, "work_width": W, "work_height": H,
            "regions": _regions_info(room), "edited": bool((room["label"] != room["label0"]).any())}

def _png(arr):
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format="PNG"); return buf.getvalue()

# ---- 연결 확인 / 바닥재 ----
@app.get("/health")
def health():
    return {"status": "ok", "rooms": len(ROOMS), "floors": len(FLOOR_LIBRARY), "model": MODEL,
            "segmenter_loaded": SEG_MODEL[0] in _seg_cache, "clip_loaded": "m" in _clip_cache}

class FloorItem(BaseModel):
    id: str
    url: Optional[str] = None
    vec: Optional[List[float]] = None       # CLIP 벡터 (선택). 미리 계산해 주면 서버가 빨리 켜짐

class FloorSync(BaseModel):
    floors: List[FloorItem]

@app.post("/floors/sync", dependencies=[Depends(check_key)])
def floors_sync(body: FloorSync):
    """백엔드의 바닥재 목록을 통째로 등록해요 (기존 목록은 바뀜). url의 사진을 내려받아 평균 색과 CLIP 벡터를 만듦."""
    with LOCK:
        try:
            lib = set_floor_library([{"id": f.id, "url": f.url, "vec": f.vec} for f in body.floors])
        except RuntimeError as e:
            raise AIError(422, "no_usable_floors", str(e))
        for room in ROOMS.values():
            room["laid"].clear()
        return {"count": len(lib), "ids": list(lib), "skipped": [f.id for f in body.floors if str(f.id) not in lib]}

@app.get("/floors", dependencies=[Depends(check_key)])
def floors_list():
    return {"count": len(FLOOR_LIBRARY), "floors": [{"id": k, "avg_hex": v["avg_hex"]} for k, v in FLOOR_LIBRARY.items()]}

# ---- 방 만들기 (영역 분할) ----
@app.post("/rooms", dependencies=[Depends(check_key)])
def create_room(image: Optional[UploadFile] = File(None), image_url: Optional[str] = Form(None)):
    """방 사진 → 영역 분할. image(파일) 또는 image_url(주소) 중 하나를 보내요."""
    if image is not None:
        data = image.file.read()
    elif image_url:
        data = _download(image_url, "방 사진")
    else:
        raise AIError(422, "invalid_request", "image(파일) 또는 image_url 중 하나가 필요해요")
    original = _pil_from_bytes(data, "방 사진")
    work = _resize_max(original, SERVER_MAX_SIDE)
    with LOCK:
        started = time.time()
        try:
            masks = {n: clean_mask(m) for n, m in segment_room(work).items()}
        except Exception as e:
            raise AIError(500, "segmentation_failed", f"영역 분할에 실패했어요 ({type(e).__name__})")
        label = np.zeros(masks["wall"].shape, np.uint8)
        for rid, key in [(3, "ceiling"), (1, "wall"), (2, "floor")]:
            label[masks[key]] = rid
        if not (label > 0).any():
            raise AIError(422, "no_room_detected", "벽이나 바닥을 찾지 못했어요. 방 안쪽이 보이는 사진인지 확인해 주세요")
        room_id = uuid.uuid4().hex[:12]
        ROOMS[room_id] = {"image": np.array(work), "label0": label.copy(), "label": label, "masks": _masks_from_label(label),
                          "orig_size": original.size, "laid": {}, "created": time.time()}
        while len(ROOMS) > MAX_ROOMS:
            ROOMS.popitem(last=False)
        out = _room_summary(room_id, ROOMS[room_id])
    out["seconds"] = round(time.time() - started, 2)
    out["warnings"] = ["벽 여러 면 분리(wall1~)와 몰딩은 아직 지원하지 않아요. 벽은 allWalls 하나로 다뤄요"]
    return out

@app.get("/rooms/{room_id}", dependencies=[Depends(check_key)])
def room_info(room_id: str):
    room = _get_room(room_id)
    return _room_summary(room_id, room)

@app.delete("/rooms/{room_id}", dependencies=[Depends(check_key)])
def room_delete(room_id: str):
    with LOCK:
        if ROOMS.pop(room_id, None) is None:
            raise AIError(404, "room_not_found", "이미 지워졌거나 없는 방이에요", {"room_id": room_id})
    return {"deleted": room_id}

@app.get("/rooms/{room_id}/regions", dependencies=[Depends(check_key)])
def room_regions(room_id: str, kind: str = "overlay"):
    """영역 그림. kind=overlay: 영역별로 색칠한 반투명 PNG / kind=ids: 영역 번호(0~3)가 값인 회색 PNG. 크기는 work_width x work_height."""
    room = _get_room(room_id)
    label = room["label"]
    if kind == "ids":
        return Response(_png(label), media_type="image/png")
    if kind != "overlay":
        raise AIError(422, "invalid_request", "kind는 overlay 또는 ids여야 해요")
    rgba = np.zeros(label.shape + (4,), np.uint8)
    for rid, color in REGION_COLORS.items():
        rgba[label == rid] = (*color, 110)
    return Response(_png(rgba), media_type="image/png")

# ---- 붓 수정 ----
class Stroke(BaseModel):
    region_id: int
    width: float = 20
    points: List[List[float]]

class EditsBody(BaseModel):
    strokes: List[Stroke]

@app.post("/rooms/{room_id}/edits", dependencies=[Depends(check_key)])
def room_edits(room_id: str, body: EditsBody):
    """붓으로 영역을 고쳐요. 좌표와 width는 원본 사진 픽셀 기준. region_id 0은 지우개(영역 없음으로 만듦)."""
    room = _get_room(room_id)
    with LOCK:
        label = room["label"].copy()
        H, W = label.shape
        ow, oh = room["orig_size"]
        sx, sy = W / ow, H / oh
        for k, st in enumerate(body.strokes):
            if st.region_id not in (0, 1, 2, 3):
                raise AIError(422, "invalid_region", "region_id는 0(지우개), 1(벽), 2(바닥), 3(천장) 중 하나여야 해요", {"stroke": k, "region_id": st.region_id})
            pts = [(p[0] * sx, p[1] * sy) for p in st.points if len(p) >= 2]
            if not pts:
                continue
            ipts = np.round(np.array(pts, np.float32)).astype(np.int32).reshape(-1, 1, 2)
            thick = max(1, int(round(st.width * (sx + sy) / 2)))
            if len(ipts) == 1:
                cv2.circle(label, (int(ipts[0, 0, 0]), int(ipts[0, 0, 1])), max(1, thick // 2), int(st.region_id), -1)
            else:
                cv2.polylines(label, [ipts], False, int(st.region_id), thick)
        room["label"] = label
        room["masks"] = _masks_from_label(label)
        room["laid"].clear()                    # 바닥 모양이 바뀌었으니 원근 계산 다시
        out = _room_summary(room_id, room)
    out["applied_strokes"] = len(body.strokes)
    return out

@app.post("/rooms/{room_id}/reset", dependencies=[Depends(check_key)])
def room_reset(room_id: str):
    room = _get_room(room_id)
    with LOCK:
        room["label"] = room["label0"].copy()
        room["masks"] = _masks_from_label(room["label"])
        room["laid"].clear()
        return _room_summary(room_id, room)

# ---- 레퍼런스 분석 ----
class RecommendBody(BaseModel):
    reference_urls: List[str]
    n_proposals: Optional[int] = None       # 시안 개수 (기본: 2단계의 N_PROPOSALS, 1~5)

@app.post("/recommend", dependencies=[Depends(check_key)])
def recommend(body: RecommendBody):
    """레퍼런스 사진 주소 1~6개 → 취향 키워드 + 시안(벽 색 + 바닥재 id). 바닥재는 /floors/sync로 등록한 목록에서 골라요."""
    if not body.reference_urls:
        raise AIError(422, "invalid_request", "reference_urls에 사진 주소가 1개 이상 필요해요")
    n = None if body.n_proposals is None else max(1, min(5, int(body.n_proposals)))
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        paths = []
        for i, url in enumerate(body.reference_urls[:MAX_REFERENCES]):
            img = _pil_from_bytes(_download(url, "레퍼런스 사진"), "레퍼런스 사진")
            path = os.path.join(tmp, f"ref{i}.jpg"); img.save(path, quality=92); paths.append(path)
        with LOCK, override_globals(N_PROPOSALS=n):
            try:
                analysis, seconds = analyze_reference(paths)
            except RuntimeError as e:
                msg = str(e)
                if "키가 없어요" in msg:
                    raise AIError(503, "llm_not_configured", msg)
                if msg.startswith("OpenAI 사용 불가"):
                    raise AIError(502, "llm_unavailable", msg)
                raise AIError(502, "llm_failed", msg)
            ref_vec = reference_floor_vector(paths) if FLOOR_FROM_REFERENCE else None
            pick_floors(analysis["proposals"], ref_vec)
    analysis["seconds"] = round(seconds, 2)
    analysis["warnings"] = (["바닥재 목록이 등록되지 않아 임시 가짜 바닥재(sample_marble)만 쓰고 있어요. 먼저 POST /floors/sync를 호출해 주세요"]
                            if list(FLOOR_LIBRARY) == ["sample_marble"] else [])
    return analysis

# ---- 시안 만들기 ----
class GenOptions(BaseModel):
    detail: float = 1.0                    # 벽의 원래 밝기 무늬 반영 (0~1.3)
    keep: Optional[float] = None           # 바닥: 원래 빛 유지 (0~1.6)
    spec: Optional[float] = None           # 바닥: 하이라이트 (0~2.5)
    reflect: Optional[float] = None        # 바닥: 반사 (0~1)
    rough: Optional[float] = None          # 바닥: 반사 흐림 (0~1)
    exposure: Optional[float] = None       # 바닥재 밝기 (0.5~1.3)
    horizon: Optional[float] = None        # 수평선 높이 (0~1, 위가 0)
    fov: Optional[float] = None
    rotate: Optional[float] = None
    tile: Optional[float] = None           # 바닥재 한 칸 크기 (이미지 가로 대비)

class GenerateBody(BaseModel):
    materials: Dict[str, Any]              # {"allWalls": {"hex": "#F3EBDD"}, "floor": {"id": "floor_0002"}} 또는 "floor": "floor_0002"
    options: GenOptions = GenOptions()
    preview: bool = False                  # true면 결과를 작게(긴 변 512px) 줄여서 보냄
    quality: int = 90                      # JPEG 품질 (50~100)

def _as_ref(value, key):
    if isinstance(value, str):
        return {"id": value}
    if isinstance(value, dict):
        ref = {k: value.get(k) for k in ("id", "url", "hex")}
        if ref["hex"] is not None and not HEX_PATTERN.match(str(ref["hex"])):
            raise AIError(422, "invalid_material", f"{key}: hex는 #RRGGBB 형식이어야 해요", {"hex": ref["hex"]})
        return ref
    raise AIError(422, "invalid_material", f"{key}: 재료는 id 문자열이나 {{id, url, hex}} 객체여야 해요")

def _wall_color(ref, key, warnings):
    if ref.get("hex"):
        return ref["hex"]
    if ref.get("url"):
        color = avg_hex(np.array(_pil_from_bytes(_download(ref["url"], "벽지 사진"), "벽지 사진")))
        warnings.append(f"{key}: 벽지 무늬는 아직 지원하지 않아 사진의 평균 색({color})으로 칠했어요")
        return color
    raise AIError(422, "material_unresolved", f"{key}: 벽 재료는 hex나 url이 필요해요 (id만으로는 AI 서버가 알 수 없어요)", {"material": ref.get("id")})

def _floor_texture(ref):
    if ref.get("id") and ref["id"] in FLOOR_LIBRARY:
        return ref["id"], get_floor_texture(ref["id"], FLOOR_LIBRARY)
    if ref.get("url"):
        key = "url:" + ref["url"]
        if key not in EXTRA_FLOORS:
            EXTRA_FLOORS[key] = _read_floor_raw({"url": ref["url"]})
            while len(EXTRA_FLOORS) > 20:
                EXTRA_FLOORS.popitem(last=False)
        return key, EXTRA_FLOORS[key]
    raise AIError(422, "material_unresolved", f"floor: 바닥재 '{ref.get('id')}'를 찾을 수 없어요. POST /floors/sync로 먼저 등록하거나 url을 같이 보내 주세요", {"material": ref.get("id")})

@app.post("/rooms/{room_id}/generate", dependencies=[Depends(check_key)])
def room_generate(room_id: str, body: GenerateBody):
    """방에 재료를 입혀요. 지원: allWalls(hex 또는 url), floor(id 또는 url). wall1~, moldingTop/Bottom은 무시하고 warnings로 알려줘요."""
    room = _get_room(room_id)
    started = time.time()
    with LOCK:
        warnings, wall_hex, floor_key, floor_tex, applied = [], None, None, None, {}
        for key, raw in body.materials.items():
            ref = _as_ref(raw, key)
            if key == "allWalls":
                wall_hex = _wall_color(ref, key, warnings); applied[key] = wall_hex
            elif key == "floor":
                floor_key, floor_tex = _floor_texture(ref); applied[key] = floor_key
            elif re.fullmatch(r"wall\d+", key):
                warnings.append(f"{key}: 벽 여러 면 분리는 아직 지원하지 않아 무시했어요 (allWalls만 적용)")
            elif key in ("moldingTop", "moldingBottom"):
                warnings.append(f"{key}: 몰딩은 아직 지원하지 않아 무시했어요")
            else:
                raise AIError(422, "unknown_region", f"알 수 없는 영역 이름이에요: {key}", {"allowed": ["allWalls", "floor", "wall1~", "moldingTop", "moldingBottom"]})
        if wall_hex is None and floor_tex is None:
            raise AIError(422, "nothing_to_apply", "적용할 수 있는 재료가 없어요 (allWalls 또는 floor가 필요해요)")
        masks = room["masks"]
        if wall_hex and not masks["wall"].any():
            warnings.append("벽 영역이 비어 있어 벽 색을 적용하지 못했어요")
        if floor_tex is not None and not masks["floor"].any():
            warnings.append("바닥 영역이 비어 있어 바닥재를 적용하지 못했어요")
        o = body.options
        with override_globals(HORIZON=o.horizon, FOV=o.fov, ROTATE=o.rotate, FLOOR_TILE=o.tile):
            laid = None
            if floor_tex is not None and masks["floor"].any():
                cache_key = (floor_key, HORIZON, FOV, ROTATE, FLOOR_TILE)
                if cache_key not in room["laid"]:
                    room["laid"][cache_key] = floor_perspective(room["image"].shape[:2], masks["floor"], floor_tex, FLOOR_TILE, HORIZON, FOV, ROTATE)
                laid = room["laid"][cache_key]
            view = {"image": room["image"], "masks": masks}
            out = recolor(view, wall_hex, None, o.detail, floor_laid=laid,
                          keep=o.keep, spec=o.spec, reflect=o.reflect, rough=o.rough, exposure=o.exposure)
    img = Image.fromarray(out)
    if body.preview:
        img = _resize_max(img, PREVIEW_SIDE)
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=max(50, min(100, body.quality)))
    return {"room_id": room_id, "image_base64": base64.b64encode(buf.getvalue()).decode(), "mime": "image/jpeg",
            "width": img.width, "height": img.height, "applied": applied, "warnings": warnings,
            "seconds": round(time.time() - started, 2)}

print("서버 코드 준비 완료. 다음 칸에서 서버를 켜요.")

In [ ]:
# 9단계: 서버 켜기 + 임시 주소(Cloudflare Tunnel) 만들기
import subprocess, re as _re, time as _time, urllib.request, stat
import uvicorn

PORT = 8000
_server = None

def start_server(port=PORT):
    """AI 서버를 이 노트북 안에서 켜요 (이미 켜져 있으면 그대로)."""
    global _server
    if _server is not None and _server.started:
        return _server
    _server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=port, log_level="warning"))
    threading.Thread(target=_server.run, daemon=True).start()
    for _ in range(100):
        if _server.started:
            break
        _time.sleep(0.1)
    print(f"서버 켜짐: http://127.0.0.1:{port}  (확인: /health, 명세: /docs)")
    return _server

def start_tunnel(port=PORT, timeout=60):
    """밖에서 접속할 수 있는 임시 주소(trycloudflare.com)를 만들어요. 서버를 다시 켜면 주소가 바뀌니 백엔드에 알려줘야 해요."""
    if not os.path.exists("cloudflared"):
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
        os.chmod("cloudflared", os.stat("cloudflared").st_mode | stat.S_IEXEC)
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    found = {"url": None}
    def drain():
        for line in proc.stdout:                       # 출력을 계속 비워줘야 멈추지 않음
            m = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m and not found["url"]:
                found["url"] = m.group(0)
    threading.Thread(target=drain, daemon=True).start()
    for _ in range(timeout * 10):
        if found["url"]:
            break
        _time.sleep(0.1)
    if not found["url"]:
        print("⚠️ 임시 주소를 못 만들었어요. 잠시 후 이 칸을 다시 실행해 보세요")
    return found["url"]

start_server()
PUBLIC_URL = start_tunnel()
if PUBLIC_URL:
    print("\n=== 백엔드에 전달할 것 ===")
    print("AI 서버 주소:", PUBLIC_URL)
    print("API 명세     :", PUBLIC_URL + "/docs")
    print("연결 테스트  : GET", PUBLIC_URL + "/health")
    print("인증         : 요청 헤더 X-AI-Key = (Colab 🔑 AI_SERVER_KEY에 저장한 값)" if AI_SERVER_KEY else "인증         : (AI_SERVER_KEY가 없어 인증 없이 열려 있음)")

In [ ]:
# 10단계: 서버 점검 (이 노트북 안에서 직접 한 바퀴 돌려보기)
from google.colab import files

def selftest(room_path, floor_urls=None, reference_urls=None, base=None):
    """방 사진 하나로 /rooms → (선택) /floors/sync, /recommend → /generate를 순서대로 불러보고 결과 이미지를 저장해요.
    floor_urls: {"floor_001": "https://..."} 형태로 바닥재 주소를 주면 /floors/sync부터 시험함."""
    base = base or f"http://127.0.0.1:{PORT}"
    headers = {"X-AI-Key": AI_SERVER_KEY} if AI_SERVER_KEY else {}
    show = lambda title, r: print(f"[{title}] {r.status_code}", (r.text[:200] if r.status_code >= 400 else "OK"))
    show("health", _requests.get(base + "/health"))
    if floor_urls:
        r = _requests.post(base + "/floors/sync", json={"floors": [{"id": k, "url": v} for k, v in floor_urls.items()]}, headers=headers); show("floors/sync", r)
    with open(room_path, "rb") as f:
        r = _requests.post(base + "/rooms", files={"image": f}, headers=headers)
    show("rooms", r); info = r.json(); room_id = info["room_id"]
    print("  영역:", info["regions"])
    materials = {"allWalls": {"hex": "#E8DCC8"}, "floor": {"id": next(iter(FLOOR_LIBRARY))}}      # 추천을 안 쓰거나 실패하면 이 기본값
    if reference_urls:
        r = _requests.post(base + "/recommend", json={"reference_urls": reference_urls}, headers=headers); show("recommend", r)
        if r.status_code == 200:
            p = r.json()["proposals"][0]
            print("  키워드:", r.json()["keywords_ko"], "| 첫 시안:", p["name"], p["wall"]["hex"], p["floor_id"])
            materials = {"allWalls": {"hex": p["wall"]["hex"]}, "floor": {"id": p["floor_id"]}}
    r = _requests.post(f"{base}/rooms/{room_id}/generate", json={"materials": materials, "options": {"reflect": 0.3}}, headers=headers); show("generate", r)
    if r.status_code == 200:
        res = r.json(); open("selftest.jpg", "wb").write(base64.b64decode(res["image_base64"]))
        print("  적용:", res["applied"], "| 경고:", res["warnings"], "|", res["seconds"], "초")
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8, 5.5)); plt.imshow(Image.open("selftest.jpg")); plt.axis("off"); plt.show()
    return room_id

print("[내 방 사진]을 올려주세요")
_room_file = list(files.upload().keys())[0]
selftest(_room_file)